In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

In [2]:
import os
import sys
import json
import logging

import torch
import transformers

from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,4,5,6"

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, highlight=False)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logging.basicConfig(
    level=logging.INFO,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)
logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(f"{transformers.__version__=}")

2026-05-26 09:15:20 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-05-26 09:15:20 __main__ INFO     transformers.__version__='5.3.0'


## Model

Load the LM under study. Uncomment to load — large model, takes time + VRAM.

In [3]:
from src.models import ModelandTokenizer

MODEL_KEY = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_KEY,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-05-26 09:15:43 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-05-26 09:15:43 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 09:15:44 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-05-26 09:15:44 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 09:15:44 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-26 09:15:44 httpx INFO     HTT

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-05-26 09:16:07 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 09:16:07 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-05-26 09:16:07 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0


## Dataset

`design_patterns.json` holds two concepts:
- `monolithic_vs_modular` — one big function vs split helpers/classes
- `design_pattern` — observer / singleton / strategy / factory / decorator / visitor / iterator / if_else / callback_list / subclassing

Samples use the new `prompt_stub` field (typically `"class "` or `""`) instead of per-language `stubs`, since these tasks aren't tied to a single function signature and we don't compile them.

In [5]:
from src.goals.concept_data import GoalConceptDataset, GoalConceptSample

DATA_PATH = "../data/goal_concepts/design_patterns.json"
ds = GoalConceptDataset.load(DATA_PATH)

from collections import Counter
print(f"{len(ds)} samples")
print("by concept:", Counter(s.concept for s in ds))
for s in ds:
    print(f"  [{s.concept:22s}] stub={s.prompt_stub!r:10s} {s.instruction[:70]}...")

10 samples
by concept: Counter({'design_pattern': 6, 'monolithic_vs_modular': 4})
  [monolithic_vs_modular ] stub=''         Write a Python program that parses a CSV file of employee records (nam...
  [monolithic_vs_modular ] stub=''         Implement a small command-line URL shortener in Python that supports t...
  [monolithic_vs_modular ] stub=''         Write a Python script that scrapes a list of webpages (URLs from stdin...
  [monolithic_vs_modular ] stub=''         In Python, implement a function that takes a directory path and prints...
  [design_pattern        ] stub='class '   Design a Python class `NotificationCenter` that lets multiple componen...
  [design_pattern        ] stub='class '   In Python, design a global application-configuration object that any m...
  [design_pattern        ] stub=''         In Python, write a payment-processing module that, given a request dic...
  [design_pattern        ] stub='class '   In Python, build a small library that constructs `Shape`

## Prompt construction

`build_prompt` mirrors `14_iter_rec_concept` but goes through `get_code_stub`, which now falls back to `prompt_stub` when no per-language stub is registered. Set `preferred_strategy=None` for the **proactive** (unbiased) prompt; set it to a concept-valid key (e.g. `"observer"`) for the **directed** prompt.

In [6]:
from src.goals.concept_data import StrategyDirective
from src.goals.functional import apply_chat_template
from src.tokens import prepare_input

_SYSTEM_PROMPT_PY = (
    "You are a helpful assistant proficient in Python. When completing code, "
    "ensure proper syntax and logical correctness. Make sure to have all codes "
    "inside the <code> </code> tags."
)

def build_prompt(
    mt,
    sample: GoalConceptSample,
    lang: str = "python",
    add_docstring: bool = False,
    without_comments: bool = False,
    preferred_strategy: StrategyDirective | None = None,
    system_prompt: str = _SYSTEM_PROMPT_PY,
) -> str:
    """Build a chat prompt for *sample*. Supports both per-language stubs and prompt_stub."""
    stub = sample.get_code_stub(lang, add_docstring=add_docstring)
    wc_cache = sample.without_comments
    ps_cache = sample.preferred_strategy
    sample.without_comments = without_comments
    sample.preferred_strategy = preferred_strategy
    try:
        prompt = apply_chat_template(
            mt,
            user_query=sample.full_instruction,
            incomplete_code=stub,
            system_prompt=system_prompt,
        )
    finally:
        sample.without_comments = wc_cache
        sample.preferred_strategy = ps_cache
    return prompt


@torch.inference_mode()
def sample_completions(
    mt,
    prompt: str,
    n: int = 8,
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_new_tokens: int = 400,
    stop_tag: str = "</code>",
) -> list[str]:
    """Generate n independent completions via sampling. Returns text continuations only."""
    inputs = prepare_input(prompts=prompt, tokenizer=mt)
    input_ids = inputs["input_ids"].to(mt._model.device).expand(n, -1)
    attention_mask = inputs["attention_mask"].to(mt._model.device).expand(n, -1)
    prompt_len = input_ids.shape[1]
    outputs = mt._model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=mt.tokenizer.eos_token_id,
    )
    results = []
    for out in outputs:
        text = mt.tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        if stop_tag in text:
            text = text[: text.index(stop_tag)]
        results.append(text.rstrip())
    return results

## Smoke test: one sample, both prompt modes

Compare the proactive (no directive) and directed (e.g. `observer`) prompts side-by-side.

In [7]:
sample = ds[4]   # NotificationCenter — observer-vs-callback_list pair

p_proactive = build_prompt(mt, sample)
p_directed  = build_prompt(mt, sample, preferred_strategy="observer")

print("=== PROACTIVE ===")
print(p_proactive)
print("\n=== DIRECTED (observer) ===")
print(p_directed)

=== PROACTIVE ===
<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Design a Python class `NotificationCenter` that lets multiple components subscribe to and react when a given `event_type` string is fired. Provide subscribe, unsubscribe, and fire methods.<|im_end|>
<|im_start|>assistant
<code>
class 

=== DIRECTED (observer) ===
<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Design a Python class `NotificationCenter` that lets multiple components subscribe to and react when a given `event_type` string is fired. Provide subscribe, unsubscribe, and fire methods. Use the **Observer** design pattern.<|im_end|>
<|im_start|>assistant
<code>
class

## Code extraction + pattern labeling

Heuristic labelers — kept inline so you can iterate on the indicators. They label each completion as a small dict of bool flags; aggregate over many samples to estimate `rate_pattern_X` per task.

**Caveats:**
- These are *surface* indicators (class names, method names, decorators). A direction trained on a labeler that fires on the token `notify(` may capture lexical surface rather than structural intent — exactly the competence-vs-intent confound from the existing pipeline. Use these labels for *task selection* first (which tasks reliably split), not as ground-truth probe labels.
- AST parse fails (incomplete generation, leftover `<code>` artifacts) → labeler falls back to regex-only.

In [9]:
import ast
import re
from dataclasses import dataclass, field, asdict
from typing import Optional


_CODE_TAG_RE = re.compile(r"<code>\s*(.*?)\s*(?:</code>|$)", re.DOTALL)

def extract_code(generation: str, prompt_stub: str = "") -> str:
    """Pull text inside <code>...</code>; fall back to raw generation. Prepend the
    stub so the AST parser sees a syntactically plausible chunk."""
    m = _CODE_TAG_RE.search(generation)
    body = m.group(1) if m else generation
    return (prompt_stub + body).strip()


@dataclass
class CodeStructure:
    """Cheap structural summary used by the per-concept labelers."""
    n_classes: int = 0
    n_top_level_defs: int = 0
    n_methods: int = 0          # def inside any class
    n_nested_defs: int = 0      # def inside another def
    class_names: list[str] = field(default_factory=list)
    method_names: list[str] = field(default_factory=list)
    function_names: list[str] = field(default_factory=list)
    decorator_uses: list[str] = field(default_factory=list)
    ast_ok: bool = True


def summarize_structure(code: str) -> CodeStructure:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        # regex fallback (very approximate)
        return CodeStructure(
            n_classes=len(re.findall(r"^class\s+(\w+)", code, re.M)),
            n_top_level_defs=len(re.findall(r"^def\s+(\w+)", code, re.M)),
            class_names=re.findall(r"^class\s+(\w+)", code, re.M),
            function_names=re.findall(r"^def\s+(\w+)", code, re.M),
            decorator_uses=re.findall(r"^\s*@(\w+)", code, re.M),
            ast_ok=False,
        )

    s = CodeStructure()
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.ClassDef):
            s.n_classes += 1
            s.class_names.append(node.name)
            for ch in ast.walk(node):
                if isinstance(ch, ast.FunctionDef):
                    s.n_methods += 1
                    s.method_names.append(ch.name)
        elif isinstance(node, ast.FunctionDef):
            s.n_top_level_defs += 1
            s.function_names.append(node.name)
            for ch in ast.walk(node):
                if isinstance(ch, ast.FunctionDef) and ch is not node:
                    s.n_nested_defs += 1

    for node in ast.walk(tree):
        for d in getattr(node, "decorator_list", []) or []:
            if isinstance(d, ast.Name):
                s.decorator_uses.append(d.id)
            elif isinstance(d, ast.Attribute):
                s.decorator_uses.append(d.attr)
    return s


# ── Per-concept labelers ─────────────────────────────────────────────────────

def label_monolithic_vs_modular(code: str) -> dict:
    s = summarize_structure(code)
    n_units = s.n_classes + s.n_top_level_defs
    is_monolithic = (n_units <= 1) and (s.n_methods + s.n_nested_defs <= 1)
    is_modular    = (s.n_classes >= 1) or (s.n_top_level_defs >= 2) or (s.n_nested_defs >= 1)
    return {
        "is_monolithic":   is_monolithic,
        "is_modular":      is_modular,
        "n_classes":       s.n_classes,
        "n_top_defs":      s.n_top_level_defs,
        "n_methods":       s.n_methods,
        "ast_ok":          s.ast_ok,
    }


_OBSERVER_HINTS  = (r"subscribe\b", r"unsubscribe\b", r"notify\b", r"\bObserver\b", r"\bSubject\b")
_SINGLETON_HINTS = (r"_instance\s*=\s*None", r"__new__\b", r"getInstance\b", r"\bSingleton\b")
_FACTORY_HINTS   = (r"\bFactory\b", r"\.create\(", r"\.make\(", r"create_\w+\(")
_VISITOR_HINTS   = (r"\bvisit_\w+", r"\bVisitor\b", r"\.accept\(")
_DECORATOR_HINTS = (r"^\s*@\w+", r"functools\.wraps", r"def wrapper\(")
_STRATEGY_HINTS  = (r"strategies\s*=\s*{", r"handlers\s*=\s*{", r"\bStrategy\b")
_ITERATOR_HINTS  = (r"__iter__\b", r"__next__\b", r"\byield\b")

_IFELSE_RE = re.compile(r"\belif\b", re.M)

def _any(patterns: tuple[str, ...], code: str) -> bool:
    return any(re.search(p, code, re.M) for p in patterns)


def label_design_pattern(code: str) -> dict:
    s = summarize_structure(code)
    flags = {
        "has_observer":      _any(_OBSERVER_HINTS, code),
        "has_singleton":     _any(_SINGLETON_HINTS, code),
        "has_factory":       _any(_FACTORY_HINTS, code),
        "has_visitor":       _any(_VISITOR_HINTS, code),
        "has_decorator":     _any(_DECORATOR_HINTS, code) or len(s.decorator_uses) > 0,
        "has_strategy_map":  _any(_STRATEGY_HINTS, code),
        "has_iterator":      _any(_ITERATOR_HINTS, code),
        "has_if_else_chain": len(_IFELSE_RE.findall(code)) >= 2,
        "has_callback_list": bool(re.search(r"(callbacks|subscribers|listeners)\s*=\s*(\[|\{)", code)),
        "n_classes":         s.n_classes,
        "ast_ok":            s.ast_ok,
    }
    return flags


_LABELERS: dict[str, callable] = {
    "monolithic_vs_modular": label_monolithic_vs_modular,
    "design_pattern":        label_design_pattern,
}

def label_completion(concept: str, code: str) -> dict:
    fn = _LABELERS.get(concept)
    if fn is None:
        raise KeyError(f"No labeler for concept={concept!r}. Registered: {list(_LABELERS)}")
    return fn(code)

## Run: sample completions for every task and label

For each task: generate N proactive completions, label each, store per-flag rates. Result is a DataFrame ready to filter on ("which tasks split cleanly along observer-vs-callback?").

In [10]:
import pandas as pd
from tqdm.auto import tqdm

N_SAMPLES = 8

raw_records = []   # one row per (sample, completion)
for sample in tqdm(ds, desc="tasks"):
    prompt = build_prompt(mt, sample)
    completions = sample_completions(mt, prompt, n=N_SAMPLES)
    for i, gen in enumerate(completions):
        code = extract_code(gen, prompt_stub=sample.prompt_stub or "")
        labels = label_completion(sample.concept, code)
        raw_records.append({
            "concept":      sample.concept,
            "task":         sample.instruction[:50] + "...",
            "completion":   i,
            "code":         code,
            **labels,
        })

raw_df = pd.DataFrame(raw_records)
raw_df.head()

tasks:   0%|          | 0/10 [00:00<?, ?it/s]

You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


,concept,task,completion,code,is_monolithic,is_modular,n_classes,n_top_defs,n_methods,ast_ok,has_observer,has_singleton,has_factory,has_visitor,has_decorator,has_strategy_map,has_iterator,has_if_else_chain,has_callback_list
0,monolithic_vs_modular,Write a Python program that parses a CSV file ...,0,import csv\n\ndef parse_csv(file_path):\n e...,False,True,0,3.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,monolithic_vs_modular,Write a Python program that parses a CSV file ...,1,import csv\nfrom collections import defaultdic...,False,True,0,2.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,monolithic_vs_modular,Write a Python program that parses a CSV file ...,2,import csv\n\ndef parse_csv(file_path):\n #...,False,True,0,2.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,monolithic_vs_modular,Write a Python program that parses a CSV file ...,3,import csv\n\ndef parse_and_validate_csv(file_...,False,True,0,3.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,monolithic_vs_modular,Write a Python program that parses a CSV file ...,4,import csv\n\ndef parse_csv(file_path):\n e...,False,True,0,4.0,0.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Aggregate: per-task rate of every boolean flag.
BOOL_COLS = [c for c in raw_df.columns if raw_df[c].dtype == bool]

rate_df = (
    raw_df.groupby(["concept", "task"])[BOOL_COLS]
    .mean()
    .round(2)
    .reset_index()
)
rate_df

,concept,task,ast_ok
0,design_pattern,Design a Python class `NotificationCenter` tha...,0.62
1,design_pattern,"In Python, add request-logging and authenticat...",0.88
2,design_pattern,"In Python, build a small library that construc...",0.62
3,design_pattern,"In Python, design a global application-configu...",1.00
4,design_pattern,"In Python, implement evaluation of a small exp...",0.88
5,design_pattern,"In Python, write a payment-processing module t...",0.88
6,monolithic_vs_modular,Implement a small command-line URL shortener i...,0.00
7,monolithic_vs_modular,"In Python, implement a function that takes a d...",0.75
8,monolithic_vs_modular,Write a Python program that parses a CSV file ...,1.00
9,monolithic_vs_modular,Write a Python script that scrapes a list of w...,1.00


## Inspect a single completion

Quick eyeball check — pick a row from `raw_df` and print the extracted code + its labels.

In [12]:
row = raw_df.iloc[0]
rich_print(f"[bold]task   :[/bold] {row['task']}")
rich_print(f"[bold]concept:[/bold] {row['concept']}")
rich_print(f"[bold]labels :[/bold] {{ {', '.join(f'{c}={row[c]}' for c in BOOL_COLS if row[c])} }}")
print("-" * 80)
print(row["code"])

task   : Write a Python program that parses a CSV file of e...

concept: monolithic_vs_modular

labels : { ast_ok=True }

--------------------------------------------------------------------------------
import csv

def parse_csv(file_path):
    employees = []
    with open(file_path, mode='r', newline='') as file:
        reader = csv.DictReader(file)
        for row in reader:
            try:
                name = row['name'].strip()
                department = row['department'].strip()
                salary = float(row['salary'].strip())
                if not name or not department or salary <= 0:
                    raise ValueError("Invalid data")
                employees.append({'name': name, 'department': department, 'salary': salary})
            except (ValueError, KeyError) as e:
                print(f"Warning: Skipping malformed row {row} due to {e}")
    return employees

def compute_department_totals_and_averages(employees):
    department_totals = {}
    department_counts = {}

    for employee in employees:
        dept = employee['department']
        salary = employee['salary']

   

## Next steps

1. **Task selection.** Keep tasks whose `rate_df` row shows a non-degenerate split (e.g. `has_observer ∈ [0.2, 0.8]`). These are the candidates where the model has a *real* choice — and therefore a stable internal commitment worth probing.
2. **Pivot detection.** For each retained task, find the earliest token position where the conditional `p(continuation | pattern A) >> p(continuation | pattern B)` — likely the first generated identifier (class name, def name, decorator).
3. **Cross-task transfer.** Mine a steering vector from one task's pivot in pattern A; patch it onto another task's pivot. If generation flips toward pattern A, that's evidence of a structural pattern direction (not just task-specific lexical bias).
4. **Lexical confound check.** Test transfer to tasks where you've masked / renamed the obvious surface tokens (`Observer` → `Subject`, `notify` → `broadcast`). If the direction still fires, it tracks intent over surface.

## Goalscope

In [17]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils

MASK_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    f"localization/{mt.name.split('/')[-1]}/lam20/head_mask.pt",
)

ckpt = torch.load(MASK_PATH, map_location="cpu")
print(f"Loaded checkpoint from {MASK_PATH}")
# print(f"  Trained for {ckpt['config']['n_epochs']} epochs  |  N={ckpt['config']['n_instances']} instances")
# print(f"  λ_sparse={ckpt['config']['lambda_sparse']}  lr={ckpt['config']['lr']}")

# Reconstruct mask object
LAYERS = ckpt["layers"]
n_layers_tested = len(LAYERS)
n_heads = ckpt["mask_values"].shape[1]
mask = HeadMask(n_layers_tested, n_heads)
with torch.no_grad():
    mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        mask.values[0:25] = 0  # zero out early layers
        mask.values[56:] = 0  # zero out late layers
    if mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
        mask.values[0:18] = 0  # zero out early layers
        mask.values[45:] = 0  # zero out late layers

threshold = 0.9
selected_heads = mask.selected_heads(LAYERS, threshold=threshold)
print(
    f"\n  Layers: {LAYERS[0]}–{LAYERS[-1]}  ({n_layers_tested} tested, {n_heads} heads each)"
)
print(f"  Active heads (mask > {threshold}): {len(selected_heads)}")
print(f"  Selected heads: {selected_heads}")

_n_lt = len(LAYERS)
_n_h = mt.config.num_attention_heads
binary_mask = HeadMask(_n_lt, _n_h)
with torch.no_grad():
    binary_mask.values.zero_()
    for _layer_idx, _head_idx in selected_heads:
        binary_mask.values[LAYERS.index(_layer_idx), _head_idx] = 1.0

print(
    f"Binary mask: {binary_mask.n_active()} active heads / {_n_lt * _n_h} total heads "
    f"({100 * binary_mask.n_active() / (_n_lt * _n_h):.2f}%)"
)

Loaded checkpoint from /disk/u/arnab/Codes/Projects/goals/results/localization/Qwen2.5-Coder-32B-Instruct/head_mask.pt

  Layers: 0–63  (64 tested, 40 heads each)
  Active heads (mask > 0.9): 122
  Selected heads: [(28, 13), (29, 17), (29, 18), (30, 8), (30, 25), (30, 35), (31, 18), (31, 32), (32, 3), (32, 13), (34, 25), (34, 30), (34, 33), (35, 13), (35, 39), (37, 13), (37, 15), (38, 0), (38, 2), (39, 4), (39, 8), (39, 9), (39, 14), (39, 31), (39, 32), (39, 34), (40, 2), (40, 8), (40, 11), (40, 14), (40, 32), (40, 34), (41, 6), (41, 9), (41, 12), (41, 13), (42, 39), (43, 5), (43, 8), (43, 13), (43, 30), (44, 12), (44, 13), (44, 14), (44, 18), (44, 38), (45, 15), (45, 17), (45, 18), (45, 21), (45, 22), (45, 31), (45, 32), (45, 33), (45, 34), (46, 1), (46, 4), (46, 7), (46, 8), (46, 9), (46, 18), (46, 19), (46, 20), (46, 25), (46, 26), (46, 35), (47, 5), (47, 6), (47, 7), (47, 8), (47, 18), (47, 19), (47, 23), (47, 30), (47, 32), (47, 35), (47, 39), (48, 25), (48, 26), (48, 28), (48, 29

In [35]:
from src.goals.goalscope import greedy_continue, apply_goalscope

sample = ds[0] 
test_prompt = build_prompt(mt, sample, without_comments=True)
test_prompt += "\ndef process_csv(file_path):"

test_gen = greedy_continue(
    mt,
    **prepare_input(prompts=test_prompt, tokenizer=mt),
    max_new_tokens=512,
)

rich_print(f"[white]{escape(test_prompt)}[/white][bright_yellow]{escape(test_gen)}[/bright_yellow]")

<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that parses a CSV file of employee records (name,department,salary), validates each row, 
computes per-department totals and averages, and prints a formatted report. Treat malformed rows as warnings, not 
fatal errors. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>

def process_csv(file_path):  
    import csv
    from collections import defaultdict
    
    departments = defaultdict(lambda: {'total': 0, 'count': 0})
    warnings = []
    
    with open(file_path, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            try:
                name = row['name'].strip()
                department = row['department'].strip()
                salary = float(row['salary'].strip())
                departments[department]['total'] += salary
                departments[department]['count'] += 1
            except (ValueError, KeyError):
                warnings.append(f"Malformed row: {row}")
    
    print("Department Report")
    print("-----------------")
    for dept, data in departments.items():
        avg_salary = data['total'] / data['count']
        print(f"{dept}: Total Salary = {data['total']:.2f}, Average Salary = {avg_salary:.2f}")
    
    if warnings:
        print("\nWarnings:")
        for warning in warnings:
            print(warning)

</code>

In [ ]:
from src.goals.goalscope import greedy_continue, apply_goalscope

sample = ds[0] 
test_prompt = build_prompt(mt, sample, without_comments=True, preferred_strategy="modular")
test_prompt += "\ndef process_csv(file_path):"

print(test_prompt)
test_mod = greedy_continue(
    mt,
    **prepare_input(prompts=test_prompt, tokenizer=mt),
    max_new_tokens=512,
)
rich_print(f"[white]{escape(test_prompt)}[/white][bright_yellow]{escape(test_mod)}[/bright_yellow]")

<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that parses a CSV file of employee records (name,department,salary), validates each row, computes per-department totals and averages, and prints a formatted report. Treat malformed rows as warnings, not fatal errors. Decompose the solution into small, well-named helper functions or classes. Each helper should do one thing. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>

def process_csv(file_path):


<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that parses a CSV file of employee records (name,department,salary), validates each row, 
computes per-department totals and averages, and prints a formatted report. Treat malformed rows as warnings, not 
fatal errors. Decompose the solution into small, well-named helper functions or classes. Each helper should do one 
thing. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>

def process_csv(file_path):  
    import csv
    from collections import defaultdict

    def read_file(file_path):
        with open(file_path, mode='r', newline='') as file:
            return list(csv.reader(file))

    def validate_row(row):
        if len(row) != 3:
            return False, "Invalid row length"
        name, department, salary = row
        if not name or not department:
            return False, "Name and department are required"
        try:
            float(salary)
        except ValueError:
            return False, "Invalid salary format"
        return True, None

    def parse_row(row):
        name, department, salary = row
        return name, department, float(salary)

    def accumulate_data(data, name, department, salary):
        data['total_employees'] += 1
        data['departments'][department]['total_salary'] += salary
        data['departments'][department]['employee_count'] += 1

    def calculate_averages(data):
        for department, details in data['departments'].items():
            details['average_salary'] = details['total_salary'] / details['employee_count']

    def format_report(data):
        report = ["Department Report"]
        report.append(f"Total Employees: {data['total_employees']}")
        for department, details in data['departments'].items():
            report.append(f"Department: {department}")
            report.append(f"  Total Salary: {details['total_salary']:.2f}")
            report.append(f"  Average Salary: {details['average_salary']:.2f}")
            report.append(f"  Employee Count: {details['employee_count']}")
        return "\n".join(report)

    rows = read_file(file_path)
    data = {'total_employees': 0, 'departments': defaultdict(lambda: {'total_salary': 0, 'employee_count': 0})}
    for row in rows[1:]:  # Skip header
        is_valid, error = validate_row(row)
        if is_valid:
            name, department, salary = parse_row(row)
            accumulate_data(data, name, department, salary)
        else:
            print(f"Warning: Skipping row due to {error}: {row}")
    calculate_averages(data)
    print(format_report(data))

</code>

: 

In [28]:
def _build_goalscope_prompt(mt) -> str:
    """Build the GoalScope destination prompt ("What are you doing at this line?")."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant proficient in programming."},
        # {"role": "user", "content": "What are you doing at this line of the code. Keep your answer as brief as possible."},        
        # {"role": "user", "content": "You are writing a code that does a specific task. What is the task? Keep your answer as brief as possible."},
        # {"role": "user", "content": "You are completing a code that implements a specific task. The code is cropped at the current line. What are the steps you **yet** need to do to complete the code? Give your answer as a concise list of steps."},
        {"role": "user", "content": """You are writing a code that does a specific task and you are about to write a specific line of the code. 3 questions for you, which you should answer in order and as concisely as possible. **Do not write any code in your answer.**:
1) What is the broader functionality you are implementing in the code?
2) Describe in natural language the specific think you are about to do **at this line**.
3) How does this line fit into the overall functionality you described in (1)?"""},
        {"role": "assistant", "content": ""},
    ]
    try:
        return mt.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=False,
            continue_final_message=True,
            tokenize=False,
            chat_template_kwargs={"enable_thinking": False},
        )
    except TypeError:
        return mt.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=False,
            continue_final_message=True,
            tokenize=False,
        )
    
goalscope_prompt = _build_goalscope_prompt(mt)
print(goalscope_prompt)

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
You are writing a code that does a specific task and you are about to write a specific line of the code. 3 questions for you, which you should answer in order and as concisely as possible. **Do not write any code in your answer.**:
1) What is the broader functionality you are implementing in the code?
2) Describe in natural language the specific think you are about to do **at this line**.
3) How does this line fit into the overall functionality you described in (1)?<|im_end|>
<|im_start|>assistant



In [31]:
res = apply_goalscope(
    test_prompt + test_gen,
    mt=mt,
    selected_heads=selected_heads,
    # strategy="replace", alpha=1.0,
    strategy="additive", alpha=0.2,
    batch_size=32,
    max_new_tokens=128,
    title=f"GoalScope (Head Patch) — {sample.function_name}",
    # n_steer_tokens=16,
    goalscope_prompt=goalscope_prompt,
)

GoalScope[additive]:   0%|          | 0/5 [00:00<?, ?it/s]